# 05. Logistic Regression: Training-Only Model Selection

Use the existing outer **training** split only. Do not load or recreate the test split.
The flow is: load training reviews -> join movie features -> grouped CV with fold-local
encoding -> compare mean validation F1 -> refit the chosen configuration.

CV selects hyperparameters, not averaged coefficient weights: each fold has its own
vocabulary. Fresh weights are learned during refitting. Spoiler cues remain unchanged;
there is no ablation in this notebook.

## 1. Setup and Search Settings

The default is a 10,000-review development sample with all eight features and five
folds. Set `sample_size = None` to tune and refit on the complete training partition.
Sampling is uniform, without replacement, before grouped CV. Results from a sample
are preliminary, not full-data or held-out test performance.

Search L1, L2 and elastic net at three C values, plus one unpenalized model; C has no
effect without regularization. Elastic net's mixing ratio is fixed at 0.5. Compare
unigrams against unigrams + bigrams. The vocabulary cap applies to each text field.
Oversampling is fixed on, for training rows only; class weighting is not also applied.

This grid has 20 configurations (100 fits). Weakly regularized L1/elastic-net
models can be slow. Edit the grids above the training cell to reduce or expand
the search; keep the same data, grouping and seed when comparing settings.

In [ ]:
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd
import sklearn

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src").is_dir(), "Start in the repository root or notebooks/."
sys.path.insert(0, str(PROJECT_ROOT))

from src.prepare_features import INITIAL_FEATURES, prepare_features
from src.split_data import RANDOM_STATE
from src.logistic_regression import tune_logistic_regression, refit_logistic_regression
from src.download_data import download_data
from src.data_cleaning import clean_data
from src.split_data import create_train_test_split, RANDOM_STATE

sample_size = 10000  # None uses the full saved training partition.
k = 5
group_by = "review_text"
selected_features = INITIAL_FEATURES.copy()
oversample_positive = True
max_iter = 1000
tol = 1e-3

model_grid = [
    {"penalty": ["l2", "l1", "elasticnet"], "C": [0.001, 0.01, 0.1, 1.0, 10]},
    {"penalty": [None]},
]
tfidf_grid = {
    "ngram_range": [(1, 1), (1, 2), (1, 3)],
    "min_df": [3],
    "max_features": [5000],
}


## 2. Load Only Training Data

Read the existing training split in chunks to avoid loading the entire file for a
small experiment. Movie metadata is joined deterministically; vocabularies, genre
columns, medians and scales are learned later, within each fold. The test file is
never opened. Keep the existing outer split fixed.

In [ ]:
dataset_files = ["IMDB_reviews.json", "IMDB_movie_details.json"]
raw_dir = PROJECT_ROOT / "data/raw"
cleaned_dir = PROJECT_ROOT / "data/cleaned"

if not all((raw_dir / name).exists() for name in dataset_files):
    download_data()
if not all((cleaned_dir / name).exists() for name in dataset_files):
    clean_data()

split_dir = PROJECT_ROOT / "data/splits"
if group_by == "movie_id":
    split_dir = split_dir / "movie_disjoint"
split_files = ["IMDB_reviews_train.json", "IMDB_reviews_test.json"]
if not all((split_dir / name).exists() for name in split_files):
    create_train_test_split(group_by=group_by)

train_path = split_dir / "IMDB_reviews_train.json"

with train_path.open() as source:
    total_training_rows = sum(1 for _ in source)
selected_rows = None
if sample_size is not None:
    selected_rows = np.random.default_rng(RANDOM_STATE).choice(
        total_training_rows, size=min(sample_size, total_training_rows), replace=False,
    )
with pd.read_json(train_path, lines=True, convert_dates=False, chunksize=10000) as reader:
    reviews = pd.concat([
        chunk if selected_rows is None else chunk.loc[chunk.index.isin(selected_rows)]
        for chunk in reader
    ])
source_row_indices = reviews.index.to_numpy()
reviews = reviews.reset_index(drop=True)

movies = pd.read_json(
    PROJECT_ROOT / "data/cleaned/IMDB_movie_details.json",
    lines=True, convert_dates=False,
)
model_data = prepare_features(reviews, movies)
model_data["is_spoiler"] = reviews["is_spoiler"].to_numpy()
model_data["movie_id"] = reviews["movie_id"].to_numpy()
print(f"Using {len(model_data):,} of {total_training_rows:,} training reviews.")
print(f"Spoiler prevalence: {model_data['is_spoiler'].mean():.2%}")
del reviews, movies


## 3. Tune Using Mean Validation F1

Every configuration uses the same grouped folds and original text. Duplicate review
groups stay together. For each TF-IDF configuration, encoders are fitted on that
fold's training rows, then training rows alone are oversampled. The encoded matrices
are reused across logistic regression candidates; only one fold is kept at a time.

The winner must converge in every fold. Precision, recall and F1 refer to the spoiler
class, using the default 0.5 probability threshold. Also report macro-F1 and
**average precision (AP)**, a non-interpolated precision-recall summary, not trapezoidal
PR-AUC. Means and standard deviations are over validation folds, not training scores.
These are tuning scores and can be optimistic after model selection.

In [ ]:
summary, fold_scores, best_params = tune_logistic_regression(
    model_data, model_grid=model_grid, tfidf_grid=tfidf_grid,
    feature_columns=selected_features, k=k, group_by=group_by,
    random_state=RANDOM_STATE, oversample_positive=oversample_positive,
    max_iter=max_iter, tol=tol,
)
display(summary)
display(fold_scores.groupby("candidate")[["iterations", "seconds"]].max())
print("Selected parameters:", best_params)


## 4. Inspect the Selected Configuration

Unconverged candidates remain visible but are excluded from selection. Increase
`max_iter` and rerun the same grid to compare them properly. Keep the primary ranking
metric fixed before comparing runs. Validation folds are not an independent final
performance estimate; the test set stays reserved until all choices are frozen.

In [ ]:
if best_params is None:
    raise RuntimeError("No candidate converged in every fold. Increase max_iter and rerun.")
best_candidate = summary.index[summary["converged"]][0]
display(summary.loc[[best_candidate]])
display(fold_scores.loc[fold_scores["candidate"] == best_candidate])


## 5. Refit and Save, Without Evaluating Test Data

Refit fresh encoders on all supplied development rows, oversample only their encoded
training matrix, then learn the selected model's coefficients. Rotating CV validation
rows can now be used for refitting because selection is complete. Any separate
held-out validation or test data must remain excluded.

With the default sample, this saves a **sample-trained development model**. Use
`sample_size = None` and rerun for a full-training model. The saved pipeline includes
all fitted encoders and the classifier; its prediction methods do not oversample.
Results and selected source-row indices are saved alongside it for reproducibility.

In [ ]:
model = refit_logistic_regression(
    model_data, best_params, feature_columns=selected_features,
    random_state=RANDOM_STATE, oversample_positive=oversample_positive,
    max_iter=max_iter, tol=tol,
)

scope = "full" if len(model_data) == total_training_rows else f"sample_{len(model_data)}"
output_dir = PROJECT_ROOT / "data/models" / f"logistic_regression_{group_by}_{scope}"
output_dir.mkdir(parents=True, exist_ok=True)
summary.to_csv(output_dir / "cv_summary.csv")
fold_scores.to_csv(output_dir / "cv_fold_scores.csv", index=False)
np.save(output_dir / "training_row_indices.npy", source_row_indices)
joblib.dump({
    "pipeline": model,
    "best_params": best_params,
    "feature_columns": selected_features,
    "training_path": str(train_path),
    "training_rows": len(model_data),
    "total_training_rows": total_training_rows,
    "k": k,
    "group_by": group_by,
    "random_state": RANDOM_STATE,
    "oversample_positive": oversample_positive,
    "remove_spoilers": False,
    "selection_metric": "mean validation f1",
    "model_grid": model_grid,
    "tfidf_grid": tfidf_grid,
    "max_iter": max_iter,
    "tol": tol,
    "sklearn_version": sklearn.__version__,
}, output_dir / "model.joblib", compress=3)
print("Saved model and CV results:", output_dir)


### Implementation Note

This project pins scikit-learn 1.9.1. Its `penalty` argument is deprecated, so the
helper maps the familiar penalty names to `l1_ratio` (0 for L2, 1 for L1, 0.5 for
elastic net) and uses `C=np.inf` for no penalty. It uses SAGA for L1/elastic net and
L-BFGS for L2/no penalty. See the [official LogisticRegression documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).

No feature elimination, spoiler-cue ablation, threshold search or test evaluation
is performed here.